# E7 Local / Colab Decider Evaluation (`#727` / `#731`, D384–D387)

**One-click Run All** notebook for evaluating local/open-weights deciders on Colab Pro:
- **Deciders:** small `Qwen-VL` (`Qwen/Qwen2.5-VL-3B-Instruct`), `laya` text-only (`convaiinnovations/laya`, with mandatory HF custom `.py` audit gate), and 1 vision-capable diffusion LM (`GSAI-ML/LLaDA-V`, with mandatory HF custom `.py` audit gate).
- **Dataset scopes:** supports `e3_16_segments` (old 16 segments, exploratory), `e6_12_segments` (E6 new 12 segments, formal comparison additional group), or `combined`.
- **Acceleration comparison:** runs **both** accelerated (CUDA/bf16) and unaccelerated (CPU, capped to `CPU_SAMPLE_CAP` decision points when slow) for each model.
- **GT Isolation (D385):** Label-only answer table is read **ONLY** in Cell 7 (Scoring Cell) and hard-fails if missing or if any forbidden GT field (`gt_visible`, `gt_boxes`, etc.) is present. Decider code (`e7_decider_eval.py`) never receives or reads answer tables.
- **Identity status:** `UNCONFIRMED`. No paid API calls.

In [ ]:
# === CELL 1: CENTRALIZED CONFIGURATION (Edit only here if needed) ===
from pathlib import Path

# Repository & pinned git commit for reproducible Colab execution
GIT_REPO_URL = "https://github.com/Polalalabear/amidst.git"
PINNED_GIT_COMMIT = "main"  # Replace with exact 40-char commit SHA when freezing a formal run

# Upload pack location (e.g., Google Drive mount on Colab)
PACK_ROOT = Path("/content/drive/MyDrive/amidst/e7-colab-pack")
OUTPUT_DIR = Path("/content/drive/MyDrive/amidst/e7-outputs")
REPO_DIR = Path("/content/amidst")

# Set SMOKE_MODE = True for offline/CI dry-run with deterministic synthetic backend (no GPU / HF download)
SMOKE_MODE = False

# Dataset scope: "e3_16_segments" | "e6_12_segments" | "combined"
DATASET_SCOPE = "e3_16_segments"

# Input modes to run per model (image_text = primary, text_only = ablation)
INPUT_MODES = ["image_text", "text_only"]

# Acceleration comparison toggles
RUN_ACCELERATED = True
RUN_UNACCELERATED = True
ACCELERATED_DTYPE = "cuda_bf16"
CPU_SAMPLE_CAP = 20  # Fixed small-sample cap for slow CPU unaccelerated comparison

# Model configurations (hf_revision="main" or exact commit SHA; runtime resolution records exact commit & weight SHA-256s)
MODEL_CONFIGS = [
    {
        "party_id": "qwen_vl",
        "adapter": "QwenVLDecider",
        "hf_repo_id": "Qwen/Qwen2.5-VL-3B-Instruct",
        "hf_revision": "main",
        "custom_code_audited": True,
        "audited_commit": None,
        "modes": ["image_text", "text_only"],
    },
    {
        "party_id": "laya_text",
        "adapter": "LayaTextDecider",
        "hf_repo_id": "convaiinnovations/laya",
        "hf_revision": "main",
        "custom_code_audited": False,  # MUST audit HF custom .py files and set True + audited_commit before running
        "audited_commit": None,
        "modes": ["text_only"],
    },
    {
        "party_id": "diffusion_vlm",
        "adapter": "DiffusionVLMDecider",
        "hf_repo_id": "GSAI-ML/LLaDA-V",
        "hf_revision": "main",
        "custom_code_audited": False,  # MUST audit HF custom .py files and set True + audited_commit before running
        "audited_commit": None,
        "modes": ["image_text", "text_only"],
    },
]

# Pinned dependency versions (D384).
# NOTE: numpy==2.1.3 and Pillow==11.1.0 match Colab's preinstalled runtime and are intentionally
# lower than pyproject.toml minimums (numpy>=2.5.3, pillow>=12.3). This does not conflict because
# the notebook adds src/ directly to sys.path without installing the amidst project wheel.
PINNED_DEPS = [
    "torch==2.5.1",
    "transformers==4.49.0",
    "accelerate==1.4.0",
    "Pillow==11.1.0",
    "numpy==2.1.3",
    "matplotlib==3.10.0",
]


In [ ]:
# === CELL 1B: MOUNT GOOGLE DRIVE (COLAB) & CLONE / CHECKOUT PINNED REPO ===
import subprocess

try:
    from google.colab import drive
    drive.mount("/content/drive")
    print("Mounted Google Drive at /content/drive")
except ImportError:
    print("Not running in Google Colab; skipping drive.mount('/content/drive') (local fallback).")

if (Path.cwd() / "src" / "amidst").is_dir():
    REPO_DIR = Path.cwd()
    print(f"Already inside repo checkout: {REPO_DIR}; skipping git clone.")
elif not (REPO_DIR / ".git").is_dir():
    print(f"Cloning {GIT_REPO_URL} -> {REPO_DIR} ...")
    subprocess.check_call(["git", "clone", GIT_REPO_URL, str(REPO_DIR)])
    subprocess.check_call(["git", "-C", str(REPO_DIR), "checkout", PINNED_GIT_COMMIT])
else:
    print(f"Repo already present at {REPO_DIR}; checking out {PINNED_GIT_COMMIT} ...")
    subprocess.check_call(["git", "-C", str(REPO_DIR), "fetch", "--all", "--tags"])
    subprocess.check_call(["git", "-C", str(REPO_DIR), "checkout", PINNED_GIT_COMMIT])


In [ ]:
# === CELL 2: INSTALL PINNED DEPENDENCIES ===
import subprocess, sys

if not SMOKE_MODE:
    print("Installing pinned dependencies:", PINNED_DEPS)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *PINNED_DEPS])
else:
    print("SMOKE_MODE=True; skipping pip install of PINNED_DEPS.")

if (REPO_DIR / "src").is_dir() and str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))


In [ ]:
# === CELL 3: HARDWARE & ENVIRONMENT PROVENANCE ===
import json
from amidst.evaluation.e7_decider_eval import collect_hardware_provenance

HARDWARE_PROVENANCE = collect_hardware_provenance(REPO_DIR if REPO_DIR.is_dir() else None)
print(json.dumps(HARDWARE_PROVENANCE, indent=2, ensure_ascii=False))


In [ ]:
# === CELL 4: UPLOAD PACK MANIFEST SHA-256 VERIFICATION (STOPS ON MISMATCH) ===
from amidst.evaluation.e7_decider_eval import verify_manifest_sha256

MANIFEST_VERIFICATION = verify_manifest_sha256(PACK_ROOT)
print("MANIFEST verified:", MANIFEST_VERIFICATION)


In [ ]:
# === CELL 5: LOAD INFERENCE REQUESTS (DECIDER SIDE — NO ANSWER TABLE READ HERE) ===
import json
from amidst.evaluation.e7_decider_eval import DeciderRequest, filter_requests_by_scope

def load_requests_jsonl(pack_root: Path, scope: str) -> list[DeciderRequest]:
    files = []
    if scope in ("e3_16_segments", "combined"):
        files.append(pack_root / "requests" / "e3_16_segments.jsonl")
    if scope in ("e6_12_segments", "combined"):
        files.append(pack_root / "requests" / "e6_12_segments.jsonl")
    out = []
    for fpath in files:
        if not fpath.is_file():
            continue
        for line in fpath.read_text(encoding="utf-8").splitlines():
            if not line.strip():
                continue
            d = json.loads(line)
            out.append(
                DeciderRequest(
                    request_id=d["request_id"],
                    case_id=d["case_id"],
                    dataset_group=d["dataset_group"],
                    tick=int(d["tick"]),
                    variable_text=d["variable_text"],
                    candidates=tuple(d.get("candidates") or []),
                    camera_ids=tuple(d.get("camera_ids") or []),
                    image_paths={k: str(pack_root / v) for k, v in (d.get("image_paths") or {}).items()},
                )
            )
    return filter_requests_by_scope(out, scope)

REQUESTS = load_requests_jsonl(PACK_ROOT, DATASET_SCOPE)
print(f"Loaded {len(REQUESTS)} inference requests for scope={DATASET_SCOPE}")


In [ ]:
# === CELL 6: RUN DECIDERS (ACCELERATED vs UNACCELERATED; REAL HF BACKENDS OR SMOKE; NO GT ACCESS) ===
from amidst.evaluation.e7_decider_eval import (
    CustomCodeAuditError,
    DiffusionVLMDecider,
    LayaTextDecider,
    QwenVLDecider,
    build_diffusion_vlm_inference_fn,
    build_laya_text_inference_fn,
    build_qwen_vl_inference_fn,
    build_smoke_inference_fn,
    resolve_hf_provenance,
    run_acceleration_comparison,
)

ADAPTER_MAP = {
    "LayaTextDecider": LayaTextDecider,
    "QwenVLDecider": QwenVLDecider,
    "DiffusionVLMDecider": DiffusionVLMDecider,
}

RAW_COMPARISONS = []
for cfg in MODEL_CONFIGS:
    cls = ADAPTER_MAP[cfg["adapter"]]
    kwargs = {"hf_repo_id": cfg["hf_repo_id"], "hf_revision": cfg["hf_revision"]}
    if cfg["adapter"] in ("LayaTextDecider", "DiffusionVLMDecider"):
        kwargs["custom_code_audited"] = cfg["custom_code_audited"]
        kwargs["audited_commit"] = cfg["audited_commit"]

    if SMOKE_MODE:
        kwargs["inference_fn"] = build_smoke_inference_fn(cfg["party_id"])
    elif cfg["adapter"] == "QwenVLDecider":
        kwargs["inference_fn"] = build_qwen_vl_inference_fn(
            cfg["hf_repo_id"],
            cfg["hf_revision"],
            use_acceleration=RUN_ACCELERATED,
        )
    elif cfg["adapter"] == "LayaTextDecider":
        kwargs["inference_fn"] = build_laya_text_inference_fn(
            cfg["hf_repo_id"],
            cfg["hf_revision"],
            custom_code_audited=cfg["custom_code_audited"],
            audited_commit=cfg["audited_commit"],
            use_acceleration=RUN_ACCELERATED,
        )
    elif cfg["adapter"] == "DiffusionVLMDecider":
        kwargs["inference_fn"] = build_diffusion_vlm_inference_fn(
            cfg["hf_repo_id"],
            cfg["hf_revision"],
            custom_code_audited=cfg["custom_code_audited"],
            audited_commit=cfg["audited_commit"],
            use_acceleration=RUN_ACCELERATED,
        )

    decider = cls(**kwargs)
    decider.validate_custom_code_gate()
    hf_prov = (
        resolve_hf_provenance(cfg["hf_repo_id"], cfg["audited_commit"] or cfg["hf_revision"])
        if not SMOKE_MODE
        else {"hf_repo_id": cfg["hf_repo_id"], "requested_revision": cfg["hf_revision"], "resolved_commit_sha": "SMOKE_MODE", "weight_files_sha256": {}}
    )
    for mode in cfg["modes"]:
        if mode not in INPUT_MODES:
            continue
        res = run_acceleration_comparison(
            decider,
            REQUESTS,
            dataset_scope=DATASET_SCOPE,
            input_mode=mode,
            accelerated_label=ACCELERATED_DTYPE,
            unaccelerated_label="cpu",
            unaccelerated_sample_cap=CPU_SAMPLE_CAP,
            hf_provenance=hf_prov,
        )
        RAW_COMPARISONS.append(res)
print(f"Completed {len(RAW_COMPARISONS)} decider comparison runs (SMOKE_MODE={SMOKE_MODE}).")


In [ ]:
# === CELL 7: SCORING CELL ONLY (ONLY CELL ALLOWED TO READ D385 LABEL-ONLY ANSWER TABLE) ===
import json
from amidst.evaluation.e7_scoring_bridge import build_e7_analysis_report, score_e7_comparison

answer_table_path = PACK_ROOT / "scoring_only" / "answer_table.json"
if not answer_table_path.is_file():
    raise FileNotFoundError(
        f"MISSING_ANSWER_TABLE: {answer_table_path} is required for Cell 7 scoring (D385). "
        "Refusing to silently fall back to an empty answer table."
    )
ANSWER_TABLE = json.loads(answer_table_path.read_text(encoding="utf-8"))

frozen_e5_path = PACK_ROOT / "baselines" / "e5_report_frozen.json"
FROZEN_E5 = json.loads(frozen_e5_path.read_text(encoding="utf-8")) if frozen_e5_path.is_file() else None

SCORED_MODELS = [score_e7_comparison(c, ANSWER_TABLE) for c in RAW_COMPARISONS]
E7_REPORT = build_e7_analysis_report(
    hardware=HARDWARE_PROVENANCE,
    manifest_verification=MANIFEST_VERIFICATION,
    scored_models=SCORED_MODELS,
    dataset_scope=DATASET_SCOPE,
    frozen_e5_report=FROZEN_E5,
)
print("Scored", len(SCORED_MODELS), "model runs. Identity status:", E7_REPORT["identity_status"])


In [ ]:
# === CELL 8: WRITE e7_analysis_<date>.json, .md, AND PNG CHARTS ===
from amidst.evaluation.e7_scoring_bridge import write_e7_outputs

OUT_PATHS = write_e7_outputs(E7_REPORT, OUTPUT_DIR)
for kind, path in OUT_PATHS.items():
    print(f"Wrote {kind}: {path}")
